# T2TT training and evaluation from an external text manifest

Use prebuilt Japanese–English JSONL from JParaCrawl or another source with `pair_id`, `split`, `ja_text`, and `en_text`. Train and dev are required. Test and audio are optional. This notebook covers text training, evaluation, and attention extraction.

Mount Drive, configure and install the environment, stage text on the VM, train or restore, check dev BLEU and examples, then optionally extract attention. Checkpoints are written to the VM and published as verified Drive ZIPs. Rerun with the same revision, settings, and destination after a disconnect.


## GitHub access

For a private repository, create a fine-grained personal access token in GitHub **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Select the repository owner, choose **Only select repositories**, select `alignment-transfer-s2ut`, and grant **Contents → Read-only**. Set an expiration date. See the [GitHub token guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In Colab, open **Secrets** (the key icon), add `GITHUB_TOKEN`, paste the token into its value, and enable **Notebook access**. The setup cell reads this secret for Git operations. Public repositories also work without a token. Successful clone/fetch and checkout in the setup cell confirm access to the requested revision.

To authenticate an additional repository, add its exact HTTPS URL to `GITHUB_AUTH_REPOS` in the settings cell and grant the same token read access to that repository. The benchmark is public and uses anonymous HTTPS by default.

After a runtime reset, use the same notebook revision, experiment settings, and Drive paths, then rerun the cells. To open the notebook after the repository becomes private, download the `.ipynb` while signed into GitHub and use **File → Upload notebook** in Colab, or open a copy saved in your Drive. Enable secret access for that notebook. The runtime token is used after the notebook opens.


In [ ]:
#@title Process logging and progress
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def graceful_sigterm(function):
    """Turn a first POSIX SIGTERM into the ordinary interruption cleanup path."""
    from functools import wraps
    import os
    import signal
    import threading

    @wraps(function)
    def wrapped(*args, **kwargs):
        if (os.name != "posix" or threading.current_thread() is not threading.main_thread()
                or signal.getsignal(signal.SIGTERM) != signal.SIG_DFL):
            return function(*args, **kwargs)
        interrupted = False

        def interrupt(signum, frame):
            nonlocal interrupted
            if not interrupted:
                interrupted = True
                raise KeyboardInterrupt("SIGTERM")

        previous = signal.signal(signal.SIGTERM, interrupt)
        try:
            return function(*args, **kwargs)
        finally:
            signal.signal(signal.SIGTERM, previous)
    return wrapped


@graceful_sigterm
def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    termination_timeout = 75 if launch_env.get("AT_LOG_WANDB_DRAIN") == "1" else 5
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=termination_timeout)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


"""Standard-library GitHub authentication, embedded before Colab installation."""


GITHUB_GIT_WORKER = r'''
import base64
import os
import subprocess
import sys
from urllib.parse import quote

redactions = []
for i in range(int(os.environ.get("GIT_CONFIG_COUNT", "0"))):
    value = os.environ.get(f"GIT_CONFIG_VALUE_{i}", "")
    if value.startswith("Authorization: Basic "):
        encoded = value.split()[-1]
        token = base64.b64decode(encoded).decode("utf-8").split(":", 1)[1]
        redactions.extend((value, encoded, token, quote(token, safe="")))
redactions = sorted(set(redactions), key=len, reverse=True)
process = None
try:
    command = sys.argv[1:]
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    if command[position] in {"fetch", "pull"}:
        remote = next((arg for arg in command[position + 1:] if not arg.startswith("-")), "origin")
        if "://" not in remote:
            checked = subprocess.run([*command[:position], "remote", "get-url", "--all", remote],
                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace")
            from urllib.parse import urlsplit
            for address in checked.stdout.splitlines():
                url = urlsplit(address)
                if url.username or url.password or url.query or url.fragment:
                    raise ValueError("unsafe stored remote")
    process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        for secret in redactions:
            line = line.replace(secret, "[REDACTED]")
        print(line, end="", flush=True)
    code = process.wait()
except Exception:
    print("Git could not complete the requested operation.", file=sys.stderr, flush=True)
    code = 1
finally:
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if code:
    print("Check the repository URL and revision. For a private repository, add "
          "GITHUB_TOKEN to Colab Secrets, enable notebook access, and grant the "
          "fine-grained token Contents: Read-only for the selected repository. "
          "Check token expiry and any organization approval.", file=sys.stderr, flush=True)
sys.exit(code)
'''


def run_github_git(command, logs, *, runner, repositories, name="git", env=None, cwd=None,
                   capture_json=False):
    """Authenticate network Git calls with URL-scoped, process-only headers."""
    import base64
    import os
    from pathlib import Path
    import re
    import sys
    from urllib.parse import urlsplit

    command = list(map(str, command))
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    operation = command[position] if position < len(command) else ""
    if operation not in {"clone", "fetch", "pull", "ls-remote"}:
        return runner(command, logs, name=name, env=env, cwd=cwd, capture_json=capture_json)
    if Path(command[0]).name.lower() not in {"git", "git.exe"}:
        raise ValueError("GitHub authentication requires a git command")
    for argument in command:
        if "://" in argument:
            url = urlsplit(argument)
            if url.username or url.password or url.query or url.fragment:
                raise ValueError("Use repository URLs without credentials, queries, or fragments")
    scopes = []
    for repository in repositories:
        url = urlsplit(str(repository))
        if (url.scheme != "https" or url.netloc != "github.com" or url.query or url.fragment
                or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", url.path)):
            raise ValueError("Authentication scopes must be plain HTTPS GitHub repository URLs")
        scopes.append(url.geturl())

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Public repositories remain accessible when Secrets are unavailable.
    if token is not None and (not isinstance(token, str) or any(ord(c) < 33 or ord(c) > 126 for c in token)):
        token = None
        raise ValueError("GITHUB_TOKEN must contain a valid GitHub token without whitespace")
    git_env = dict(os.environ if env is None else env)
    for key in list(git_env):
        if (key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                or key in {"GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS", "GIT_CURL_VERBOSE", "GITHUB_TOKEN", "GH_TOKEN"}):
            git_env.pop(key)
    git_env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never", GIT_ASKPASS="", GIT_TRACE_REDACT="1")
    settings = [("credential.helper", ""), ("http.extraHeader", ""), ("http.followRedirects", "false")]
    header = None
    try:
        if token:
            header = "Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
            settings.extend(("http." + scope + ".extraHeader", header) for scope in dict.fromkeys(scopes))
        git_env["GIT_CONFIG_COUNT"] = str(len(settings))
        for i, (key, value) in enumerate(settings):
            git_env[f"GIT_CONFIG_KEY_{i}"] = key
            git_env[f"GIT_CONFIG_VALUE_{i}"] = value
        return runner([sys.executable, "-c", GITHUB_GIT_WORKER, *command], logs,
                      name=name, env=git_env, cwd=cwd, capture_json=capture_json)
    except Exception:
        raise RuntimeError("GitHub access failed. Check the logged URL/revision error and Colab Secrets "
                           "GITHUB_TOKEN access, selected repositories, Contents: Read-only, and expiry.") from None
    finally:
        token = header = None
        settings.clear()
        git_env.clear()


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    if command and command[0] == 'git':
        repositories = [globals().get('REPO_URL', 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'),
                        *globals().get('GITHUB_AUTH_REPOS', [])]
        return run_github_git(command, LOGS, runner=run_process, repositories=repositories,
                              name=name, capture_json=capture_json, env=env, cwd=cwd)
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/jparacrawl_t2tt/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


## 1. Inputs and output paths

Set `SOURCE_MANIFEST` to the external JSONL in Drive and keep it read-only. Choose a new `RUN_ID` for a new experiment and separate stages for different tokenizers. Set `RUN_TRAINING=False` to restore and evaluate a completed run.


In [ ]:
#@title Inputs, experiment, and training settings
from pathlib import Path
import os, sys, json
SOURCE_MANIFEST = '/content/drive/MyDrive/SET_TEXT_MANIFEST.jsonl' #@param {type:"string"}
RUN_ID = 'jparacrawl-t2tt-001' #@param {type:"string"}
TOKENIZER = 'sentencepiece' #@param ['character', 'sentencepiece']
MAX_UPDATES = 200000 #@param {type:"integer"}
RUN_TRAINING = True #@param {type:"boolean"}
RUN_ATTENTION = False #@param {type:"boolean"}
ALL_LAYERS = True #@param {type:"boolean"}
SNAPSHOT_UPDATES = 10000 #@param {type:"integer"}
KEEP_LAST = 2 #@param {type:"integer"}
ROOT = Path('/content/alignment-transfer-work') / RUN_ID
PERSISTENT = Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / RUN_ID
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '6fa07bc8ca8a1089f659df22fa7e5d1228362a09'
REPO = Path('/content/alignment-transfer-s2ut')
PYTHON = '/content/alignment-venv/bin/python'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '4361e55a99c7f4ea3a00c5c138f6820c34a07d06'
os.environ['MPLBACKEND'] = 'Agg'
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in ('.','..'):
    raise ValueError('RUN_ID must be a single new directory name')
if TOKENIZER not in ('character','sentencepiece') or MAX_UPDATES < 1:
    raise ValueError('invalid tokenizer or update budget')
if 'SET_' in SOURCE_MANIFEST:
    raise ValueError('Set SOURCE_MANIFEST to the text JSONL file.')
LOGS = ROOT / 'execution'
MANIFEST = ROOT / 'text-manifest/manifest.jsonl'
CONFIG = REPO / 'configs/t2tt/fisher.yaml'
TRAIN_STAGE = 't2tt-' + TOKENIZER
WORK = ROOT / TRAIN_STAGE
BACKUP = PERSISTENT / 'stages' / TRAIN_STAGE
CHECKPOINT = WORK / 'checkpoint_best.pt'
EVAL_ROOT = ROOT / ('dev-' + TOKENIZER)
EVAL_BACKUP = PERSISTENT / 'stages' / ('dev-' + TOKENIZER)
DEV_METRICS = EVAL_ROOT / 'dev-text.json'
ATTENTION = ROOT / ('attention-' + TOKENIZER + ('-all-layers' if ALL_LAYERS else ''))
ATTENTION_BACKUP = PERSISTENT / 'stages' / ATTENTION.name
print('VM:', ROOT, '\nDrive:', PERSISTENT, '\nmanifest:', MANIFEST)


## 2. Set up and check the environment

Install Python 3.10.22 and the pinned fairseq and benchmark runtime. Initial setup can take time. The logger records startup, PID, live output, ten-second liveness notices, and exit status for every command.


In [ ]:
#@title Set up Python and fairseq
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title Install the pinned benchmark runtime
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME))
execute('git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION, name='fetch-benchmark-runtime')
execute('git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION, name='checkout-benchmark-runtime')
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark', '--cuda', name='environment')
execute(PYTHON, '-m', 'pytest', 'tests/test_text_manifest.py', 'tests/test_text_t2tt.py', '-q', '--basetemp', '/content/text-t2tt-tests', name='text-cpu-preflight')


## 3. Stage text on the VM

Read the original manifest once and write the four text fields locally in their original order. Preserve text and splits and omit audio paths, hashes, and other fields.

Adding audio columns later preserves the same local manifest and identity if the four text fields match. Use a new `RUN_ID` when text or splits change.


In [ ]:
prepared = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest', '--input', SOURCE_MANIFEST, '--output', str(MANIFEST), '--resume', name='prepare-external-text')
print(json.dumps(prepared, ensure_ascii=False, indent=2))


## 4. Train or restore T2TT

Fit the vocabulary on train and select the best checkpoint by dev loss. SentencePiece uses separate Japanese and English Unigram models, each targeting 8000 pieces, with identity normalization. Character tokenization is also available. Start new training from a fresh model.

The first run starts at zero. Resume restores this stage's last checkpoint, optimizer, RNG, and cursor. Defaults are 200k updates, batch 8×4, max tokens 20000, validation every 1000, and local saves every 100. Drive publishes every 10k, at milestones, and at completion, retaining the latest two and milestones. `[T2TT loss]` reports VM observations and `durable_updates` reports saved updates.

Additional cells wait while training runs. Use `metrics-summary.json` in a snapshot folder to inspect loss history.


In [ ]:
if RUN_TRAINING:
    training_args = ['--manifest', str(MANIFEST), '--config', str(CONFIG), '--output', str(WORK),
        '--backend', 'fairseq', '--device', 'cuda', '--tokenizer', TOKENIZER, '--max-updates', str(MAX_UPDATES),
        '--batch-size', '8', '--update-frequency', '4', '--max-tokens', '20000', '--loader-workers', '2',
        '--validation-interval', '1000', '--save-interval-updates', '100', '--resume-if-exists']
    if TOKENIZER == 'sentencepiece':
        training_args += ['--source-vocab-size', '8000', '--target-vocab-size', '8000', '--sentencepiece-model-type', 'unigram']
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', TRAIN_STAGE, '--logs', str(LOGS),
        '--monitor-gpu', '--', PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'train',
        '--work', str(WORK), '--backup', str(BACKUP), '--snapshot-updates', str(SNAPSHOT_UPDATES), '--keep-last', str(KEEP_LAST),
        '--', PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', *training_args, name='text-t2tt-training')
else:
    log_event(TRAIN_STAGE, 'skipped', LOGS, 'training disabled; restore completed checkpoint below')


## 5. Dev BLEU and translation examples

A fresh VM restores from verified ZIPs. Check completed `MAX_UPDATES` and the input text hash, then evaluate the best checkpoint. The defaults are beam 5, length penalty 1, and the shared SacreBLEU settings. Generation takes source text. Results, prediction JSONL, and cache are saved in a separate verified stage.


In [ ]:
restored = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
    '--work', str(WORK), '--backup', str(BACKUP), '--expected-updates', str(MAX_UPDATES), name='restore-text-t2tt')
status = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'status', '--work', str(WORK),
    '--target-updates', str(MAX_UPDATES), name='verify-text-t2tt-complete')
print(status)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(EVAL_ROOT), '--backup', str(EVAL_BACKUP), '--',
    PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', str(MANIFEST), '--checkpoint', str(CHECKPOINT),
    '--split', 'dev', '--beam-size', '5', '--length-penalty', '1', '--device', 'cuda', '--output', str(DEV_METRICS), '--resume', name='text-dev-bleu')
metrics = json.loads(DEV_METRICS.read_text(encoding='utf-8'))
print(json.dumps({k:v for k,v in metrics.items() if k not in ('evaluation_contract',)}, ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
from IPython.display import display, Image
predictions = [json.loads(line) for line in DEV_METRICS.with_name(DEV_METRICS.stem + '-predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
references = {row['pair_id']:row['en_text'] for row in (json.loads(line) for line in MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()) if row['split']=='dev'}
display(pd.DataFrame([{'pair_id':r['pair_id'], 'JA':r['source_text'], 'Reference EN':references[r['pair_id']],
    'T2TT EN':r['hypothesis'], 'EOS':r['eos_reached']} for r in predictions[:10]]))
diagnostic = WORK / 'tokenization.json'
if diagnostic.is_file():
    tokenization = json.loads(diagnostic.read_text(encoding='utf-8'))
    print('actual vocab sizes:', tokenization['source']['actual_vocab_size'], tokenization['target']['actual_vocab_size'])
    display(pd.DataFrame(tokenization['statistics']).T)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'report', '--work', str(WORK),
    '--output', str(WORK / 'diagnostics'), name='text-learning-curve')
display(Image(filename=str(WORK / 'diagnostics/learning-curve.png')))


## 6. Optional text attention extraction

After reviewing translation quality, set `RUN_ATTENTION=True`. The text-only stage captures all layers in one forward pass when selected and saves subword spans in original codepoint coordinates. Add audio to the same IDs, splits, and text later and prepare CTC and units to project it into a speech teacher.


In [ ]:
if RUN_ATTENTION:
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(ATTENTION), '--backup', str(ATTENTION_BACKUP), '--',
        PYTHON, '-m', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', str(MANIFEST), '--checkpoint', str(CHECKPOINT),
        '--output', str(ATTENTION), '--device', 'cuda', '--resume', *(['--all-layers'] if ALL_LAYERS else []), name='text-attention')
else:
    log_event('text-attention', 'skipped', LOGS, 'set RUN_ATTENTION=True after reviewing dev translations')


## Storage and recovery

- Original manifest, read-only `SOURCE_MANIFEST`
- Canonical text in `ROOT/text-manifest/manifest.jsonl`
- Last and best checkpoints, model, and metrics under `ROOT/t2tt-<tokenizer>/`
- Checkpoint ZIPs in `PERSISTENT/stages/t2tt-<tokenizer>/`
- Separate dev and attention stages

After a runtime disconnect, rerun from the top with the same settings and revision. Unchanged text produces the same canonical bytes. Updates after the last verified snapshot are recomputed. Use a new `RUN_ID` for changed corpora, splits, or settings. [API, identity, and compatibility](../../docs/TEXT_ONLY_T2TT.md)
